## Top features: `topk_rf`

Fit on train and rank **RandomForest** `feature_importances_` on scaled features (sensors, `hotelling_t2`, hub interactions, aux).


In [ ]:
import importlib
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Reload after editing scripts/ (avoids stale pipeline modules in kernel).
import scripts.benchmark_models as _bm
import scripts.hub_interactions as _hi
import scripts.secom_pipelines as _sp

importlib.reload(_hi)
importlib.reload(_sp)
importlib.reload(_bm)

from scripts.benchmark_models import (
    BENCHMARK_RESULTS_PATH,
    build_benchmark_pipelines,
    run_holdout_benchmark,
    run_pipeline_benchmark,
    save_benchmark_results,
)
from scripts.secom_utils import load_all_tuned_params
from scripts.secom_pipelines import (
    TARGET_COL,
    feature_columns,
    load_mart,
    split_train_test,
)



In [ ]:
tuned = load_all_tuned_params()
for model_id, payload in tuned.items():
    summary = payload.get("cv_summary", {})
    pr = summary.get("mean_pr_auc", "n/a")
    thr = payload.get("classifier_threshold", "n/a")
    print(
        model_id,
        f"mean_pr_auc={pr}",
        f"threshold={thr}",
        payload.get("grid_search_best_params", {}),
    )

In [ ]:
df = load_mart()
feature_cols = feature_columns(df)
train_df, test_df = split_train_test(df)
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].astype(int)
X_test = test_df[feature_cols]
y_test = test_df[TARGET_COL].astype(int)
print(len(X_train), len(X_test), y_train.mean())

In [ ]:
pipelines = build_benchmark_pipelines(tuned)
list(pipelines.keys())

In [ ]:
leaderboard = run_pipeline_benchmark(pipelines, X_train, y_train, show_progress=True)
display(leaderboard)

In [ ]:
holdout = run_holdout_benchmark(
    pipelines, X_train, y_train, X_test, y_test, show_progress=True
)
display(holdout)

## Holdout bootstrap CIs

Point estimates on the full test split; **median** and **95% percentile interval** from stratified bootstrap (no model refit per draw).

In [ ]:
BOOTSTRAP_CI_COLS = [
    "pipeline",
    "pr_auc",
    "pr_auc_median",
    "pr_auc_ci_low",
    "pr_auc_ci_high",
    "ber_percent",
    "ber_percent_median",
    "ber_percent_ci_low",
    "ber_percent_ci_high",
]
holdout_bootstrap = holdout[
    [c for c in BOOTSTRAP_CI_COLS if c in holdout.columns]
].copy()
display(holdout_bootstrap.round(3))

In [ ]:
COMPARISON_COLS = [
    "pipeline",
    "mean_pr_auc",
    "pr_auc",
    "pr_auc_ci_low",
    "pr_auc_ci_high",
    "mean_roc_auc",
    "roc_auc",
    "mean_ber_percent",
    "ber_percent",
    "ber_percent_ci_low",
    "ber_percent_ci_high",
]
_merged = leaderboard.merge(holdout, on="pipeline", how="inner")
comparison = (
    _merged[[c for c in COMPARISON_COLS if c in _merged.columns]]
    .rename(
        columns={
            "mean_pr_auc": "pr_auc_cv",
            "pr_auc": "pr_auc_holdout",
            "pr_auc_ci_low": "pr_auc_holdout_ci_low",
            "pr_auc_ci_high": "pr_auc_holdout_ci_high",
            "mean_roc_auc": "roc_auc_cv",
            "roc_auc": "roc_auc_holdout",
            "mean_ber_percent": "ber_cv",
            "ber_percent": "ber_holdout",
            "ber_percent_ci_low": "ber_holdout_ci_low",
            "ber_percent_ci_high": "ber_holdout_ci_high",
        }
    )
    .sort_values("pr_auc_cv", ascending=False)
)
display(comparison)

In [ ]:
save_benchmark_results(
    tuned,
    leaderboard,
    holdout,
    train_rows=len(train_df),
    test_rows=len(test_df),
)
print(f"Wrote {BENCHMARK_RESULTS_PATH}")

## Top features: `topk_rf`

Fit on train and rank **RandomForest** `feature_importances_` on scaled features (sensors, `hotelling_t2`, hub interactions, aux).


In [ ]:
import numpy as np
import pandas as pd

from scripts.secom_utils import fitted_base_classifier

PIPELINE_NAME = "topk_rf"
topk_pipe = pipelines[PIPELINE_NAME]
topk_pipe.fit(X_train, y_train)

feature_names = topk_pipe.named_steps["scale"].get_feature_names_out()
importances = fitted_base_classifier(topk_pipe).feature_importances_

top_features_rf = (
    pd.DataFrame(
        {
            "feature": feature_names,
            "importance": importances,
        }
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
top_features_rf["importance_pct"] = (
    100 * top_features_rf["importance"] / top_features_rf["importance"].sum()
)
display(top_features_rf.head(50))


## Top features: `linear_lr`

Fit on train and rank **elastic-net logistic** coefficients (`|coef_|`) after preprocess + scale.


In [ ]:
import numpy as np
import pandas as pd

from scripts.secom_utils import fitted_base_classifier

PIPELINE_NAME = "linear_lr"
lr_pipe = pipelines[PIPELINE_NAME]
lr_pipe.fit(X_train, y_train)

feature_names = lr_pipe.named_steps["scale"].get_feature_names_out()
coefs = fitted_base_classifier(lr_pipe).coef_.ravel()
if len(feature_names) != len(coefs):
    raise ValueError(f"feature names ({len(feature_names)}) != coefs ({len(coefs)})")

top_features_lr = (
    pd.DataFrame(
        {
            "feature": feature_names,
            "coefficient": coefs,
            "abs_coefficient": np.abs(coefs),
        }
    )
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
)
top_features_lr["abs_coef_pct"] = (
    100 * top_features_lr["abs_coefficient"] / top_features_lr["abs_coefficient"].sum()
)

n_nonzero = int((top_features_lr["coefficient"] != 0).sum())
print(
    f"{PIPELINE_NAME}: {n_nonzero} / {len(top_features_lr)} non-zero coefs (train fit)"
)
display(top_features_lr.head(50))